# Community Optimization Strategies & Analysis
This notebook merges test and comparison workflows for different optimization strategies (pFBA, L2 norm, and custom cooperative tradeoffs) on SynCom models.

## 1. Imports, Directories and Functions

In [ ]:
import sys
sys.path.insert(0, '/home/emma/Dokumente/thesis')
from micom import Community
import pandas as pd
import os
import logging
from itertools import chain
from optlang.symbolics import Zero
from cobra.util.context import get_context
from micom.logger import logger
from functools import partial
from functions import *
from micom.solution import solve, add_pfba_objective, optimize_with_retry
from micom.util import check_modification, interface_to_str, _format_min_growth, _apply_min_growth
from micom.problems import regularize_l2_norm
from micom.community import cooperative_tradeoff
from collections.abc import Sized
import numpy as np
import matplotlib.pyplot as plt
import re
import seaborn as sns

In [ ]:
model_dir1 = "./final/HvSC1"
model_dir2 = "./final/HvSC2"
model_dir = "/home/emma/Dokumente/thesis/Model_generation_curation/Curated_models/final"
save_dir = "/home/emma/Dokumente/thesis/communities/simulations/pfba"
save_fig_dir = "/home/emma/Dokumente/thesis/plotsnfigs"

media_dir = "./media_creation/created_media"
taxonomy_df = pd.read_csv("./communities/SynComs_taxonomy.csv")

In [ ]:
# define fontsizes and colors
axis_labels = 22
title_size = 26
subtitle_size = 24
legend_size = 22
tick_size = 20

hvsc1_color = "#F2B342"
hvsc2_color = "#317EC2"
shared_color = "#24C136"
color_dict = {"HvSC1": hvsc1_color, "HvSC2": hvsc2_color}

### Helper Functions

In [ ]:
def create_medium(media_df):
    media_dict = dict(zip(media_df.iloc[:,0], media_df.iloc[:,1]))
    return media_dict

In [ ]:
def cooperative_tradeoff_nocross(community, min_growth, fraction, fluxes, pfba, atol, rtol):
    """Find the best tradeoff between community and individual growth without crossover."""
    with community as com:
        solver = interface_to_str(community.problem)
        check_modification(community)
        min_growth = _format_min_growth(min_growth, community.taxa)
        _apply_min_growth(community, min_growth)

        com.objective = com.scale * com.variables.community_objective
        min_growth = (
            optimize_with_retry(com, message="could not get community growth rate.")
            / com.scale
        )
        if not isinstance(fraction, Sized):
            fraction = [fraction]
        else:
            fraction = np.sort(fraction)[::-1]

        regularize_l2_norm(com, 0.0)
        results = []
        for fr in fraction:
            com.variables.community_objective.lb = fr * min_growth
            com.variables.community_objective.ub = min_growth
            sol = solve(community, fluxes=fluxes, pfba=pfba, atol=atol, rtol=rtol)
            results.append((fr, sol))
        if len(results) == 1:
            return results[0][1]
        return pd.DataFrame.from_records(results, columns=["tradeoff", "solution"])

In [ ]:
def get_biomass_objectives_and_define_as_constraint(community):
    biomass_obj = []
    coefficients = dict()
    n_taxa = len(community.taxonomy)
    
    # get all biomass reactions
    for rec in community.reactions:
        if rec.id.startswith("Growth"):
            biomass_obj.append(rec)
            
    # coefficient scaled to abundance -> for equal abundance = 1/community_size
    for rxn in biomass_obj:
        coefficients[rxn.forward_variable] = 1/n_taxa
        coefficients[rxn.reverse_variable] = -1/n_taxa

    constraint = community.problem.Constraint(0, lb = 0, ub = None)
    community.add_cons_vars(constraint)
    community.solver.update()
    constraint.set_linear_coefficients(coefficients = coefficients)
    return constraint

In [ ]:
def add_pfba_objective_totalcom(community, minimal_growth, constraint, atol=1e-4, rtol=1e-4):
    """Add pFBA objective."""
    constraint.lb = (1-rtol) * minimal_growth - atol
    if community.solver.objective.name == "_pfba_objective":
        raise ValueError("model already has pfba objective")
    reaction_variables = (
        (rxn.forward_variable, rxn.reverse_variable) for rxn in community.reactions
    )
    variables = chain(*reaction_variables)
    community.objective = Zero
    community.objective_direction = "min"
    community.objective.set_linear_coefficients(dict.fromkeys(variables, 1.0))
    if community.modification is None:
        community.modification = "pFBA"
    else:
        community.modification += " and pFBA :)"
    community.solver.update()

In [ ]:
def run_fba(community, fractions, fluxes):
    results = []
    opt_sol = community.optimize()
    for fr in fractions:
        with community:
            minimal_growth = opt_sol.growth_rate * fr
            biomass_constraint = get_biomass_objectives_and_define_as_constraint(community)
            add_pfba_objective_totalcom(community, minimal_growth, constraint = biomass_constraint, atol=1e-6, rtol=1e-6)

            community.solver.problem.parameters.advance.set(0)
            community.solver.problem.cleanup(1e-10)

            sol_pfba2 = community.optimize(fluxes=fluxes, raise_error = False)
            results.append((fr, sol_pfba2))

            print("---------")
            print(f"Tradeoff-value: {fr}")
            if sol_pfba2 != None:
                print(f"Community {community.solver.variables.community_objective.primal}")
                print(f"pFBA solution: {sol_pfba2.objective_value}")
            else:
                print(f"Solver status: {sol_pfba2}")
            print("---------")

    results_df = pd.DataFrame.from_records(results, columns=["tradeoff", "solution"])
    return results_df

In [ ]:
def cooperative_tradeoff_custom(community, min_growth, fraction, fluxes=True, rtol=1e-4, atol=1e-4):
    """Find the best tradeoff between community and individual growth."""
    check_modification(community)
    if not isinstance(fraction, Sized):
        fraction = [fraction]
    else:
        fraction = np.sort(fraction)[::-1]

    results = []
    for fr in fraction:
        with community as com:
            regularize_l2_norm(com, 0.0)
            minimal_growth = min_growth * fr
            biomass_constraint = get_biomass_objectives_and_define_as_constraint(com)
            biomass_constraint.lb = (1 - rtol) * minimal_growth - atol
            com.solver.update()

            com.solver.problem.parameters.advance.set(0)
            com.solver.problem.cleanup(1e-10)

            sol = solve(com, fluxes=fluxes, pfba=False, atol=atol, rtol=rtol)
            results.append((fr, sol))

    return pd.DataFrame.from_records(results, columns=["tradeoff", "solution"])

In [ ]:
def fix_individual_growth_rates(community, growth_rates, atol=1e-6, rtol=1e-6):
    """
    Add one constraint per taxon, fixing its individual biomass (Growth) flux
    close to the value found in `growth_rates` (e.g. from an L2 solution).
    """
    constraints = []
    for rxn in community.reactions:
        if not rxn.id.startswith("Growth"):
            continue

        taxon = getattr(rxn, "community_id", None)
        if taxon is None:
            taxon = rxn.id.split("Growth", 1)[1].lstrip("_")

        if taxon not in growth_rates:
            continue

        g = growth_rates[taxon]
        if pd.isna(g):
            continue

        lb = (1 - rtol) * g - atol
        ub = (1 + rtol) * g + atol
        lb, ub = min(lb, ub), max(lb, ub)

        constraint = community.problem.Constraint(
            rxn.forward_variable - rxn.reverse_variable,
            lb=lb,
            ub=ub,
            name=f"fix_growth_{taxon}",
        )
        community.add_cons_vars(constraint)
        constraints.append(constraint)

    community.solver.update()
    return constraints

In [ ]:
def run_l2_then_pfba(community, fractions, fluxes=True, atol=1e-4, rtol=1e-4):
    results = []
    opt_sol = community.optimize()
    print("optimal solution:", opt_sol.growth_rate)
    for fr in fractions:
        with community:
            minimal_growth = opt_sol.growth_rate * fr

            # --- Stage 1: L2-regularized growth distribution ---
            regularize_l2_norm(community, 0.0)

            biomass_constraint = get_biomass_objectives_and_define_as_constraint(community)
            biomass_constraint.lb = (1 - rtol) * minimal_growth - atol
            community.solver.update()

            community.solver.problem.parameters.advance.set(0)
            community.solver.problem.cleanup(1e-10)

            sol_l2 = solve(community, fluxes=False, pfba=False, atol=atol, rtol=rtol)

            if sol_l2 is None or getattr(sol_l2, "members", None) is None:
                print(f"Tradeoff {fr}: L2 stage failed, skipping")
                results.append((fr, None))
                continue

            members = sol_l2.members
            if "growth_rate" not in members.columns:
                growth_col = [c for c in members.columns if "growth" in c.lower()]
                if not growth_col:
                    print(f"Tradeoff {fr}: no growth_rate column in L2 solution.members, skipping")
                    results.append((fr, None))
                    continue
                members = members.rename(columns={growth_col[0]: "growth_rate"})

            growth_rates = members["growth_rate"].dropna().to_dict()
            growth_rates.pop("medium", None)
            # --- Stage 2: fix individual growth rates, then switch to pFBA ---
            fix_individual_growth_rates(community, growth_rates, atol=atol, rtol=rtol)

            add_pfba_objective_totalcom(
                community, minimal_growth, constraint=biomass_constraint, atol=atol, rtol=rtol
            )

            community.solver.problem.parameters.advance.set(0)

            sol_pfba = community.optimize(fluxes=fluxes, raise_error=False)
            results.append((fr, sol_pfba))

            print("---------")
            print(f"Tradeoff-value: {fr}")
            if sol_pfba is not None:
                print(f"Community objective: {community.solver.variables.community_objective.primal}")
                print(f"pFBA solution (total flux): {sol_pfba.objective_value}")
            else:
                print(f"Solver status: {sol_pfba}")
                print(community.solver.status)
            print("---------")

    results_df = pd.DataFrame.from_records(results, columns=["tradeoff", "solution"])
    return results_df

In [ ]:
def save_tradeoff_results(results_df, save_dir, prefix):
    """Saves community growth rates, individual taxon growth rates, and flux tables from tradeoff results."""
    save_path = save_dir
    summary_records = []
    taxon_growth_list = []

    for _, row in results_df.iterrows():
        tradeoff = row["tradeoff"]
        solution = row["solution"]

        if solution is None:
            continue

        com_growth = getattr(solution, "growth_rate", None)
        obj_val = getattr(solution, "objective_value", None)

        summary_records.append(
            {
                "tradeoff": tradeoff,
                "community_growth": com_growth,
                "objective_value": obj_val,
                "status": getattr(solution, "status", "unknown"),
            }
        )

        if hasattr(solution, "members") and solution.members is not None:
            members_df = solution.members.copy()
            if isinstance(members_df, pd.DataFrame):
                members_df["tradeoff"] = tradeoff
                taxon_growth_list.append(members_df)
                tradeoff_str = f"{int(tradeoff * 10):02d}"
                members_df.to_csv(os.path.join(save_path, f"{prefix}_{tradeoff_str}_individualtaxon_growth.csv"))

        if hasattr(solution, "fluxes") and solution.fluxes is not None:
            fluxes_df = solution.fluxes.copy()
            if isinstance(fluxes_df, pd.DataFrame):
                fluxes_df["tradeoff"] = tradeoff
                tradeoff_str = f"{int(tradeoff * 10):02d}"
                fluxes_df.to_csv(os.path.join(save_path, f"{prefix}_{tradeoff_str}_fluxes.csv"))

    if summary_records:
        summary_df = pd.DataFrame(summary_records)
        summary_df.to_csv(os.path.join(save_path, f"{prefix}_community_growth_summary.csv"), index=False)

    if taxon_growth_list:
        all_taxon_growth = pd.concat(taxon_growth_list, ignore_index=False)

    print(f"Results successfully saved to: {save_path}")

In [ ]:
def compare_growth_methods(
    com_name: str,
    save_dir: str,
    methods: list[str] = None,
    normalize_by_taxa: bool = True,
):
    summary_files = [
        f
        for f in os.listdir(save_dir)
        if f.startswith(com_name) and f.endswith("_community_growth_summary.csv")
    ]

    summary_dfs = []
    for file in summary_files:
        method = file[len(com_name) + 1 : -len("_community_growth_summary.csv")]

        if methods and method not in methods:
            continue

        path = os.path.join(save_dir, file)
        df = pd.read_csv(path)

        if {"tradeoff", "community_growth"}.issubset(df.columns):
            df["method"] = method
            summary_dfs.append(df)

    if not summary_dfs:
        raise FileNotFoundError(
            f"No valid community summary files found for '{com_name}' in {save_dir}"
        )

    summary_df = pd.concat(summary_dfs, ignore_index=True)
    summary_df = summary_df.sort_values(["tradeoff", "method"]).reset_index(drop=True)

    individual_files = [
        f
        for f in os.listdir(save_dir)
        if f.startswith(com_name) and f.endswith("_individualtaxon_growth.csv")
    ]

    method_pattern = (
        r"(?P<method>" + "|".join(map(re.escape, methods)) + r")"
        if methods
        else r"(?P<method>.+?)"
    )
    pattern = re.compile(
        rf"^{re.escape(com_name)}_{method_pattern}_(?P<tradeoff>\d{{2}})_individualtaxon_growth\.csv$"
    )

    taxon_growth_dfs = []
    for file in individual_files:
        match = pattern.match(file)
        if not match:
            continue

        method = match.group("method")
        tradeoff = int(match.group("tradeoff")) / 10.0
        path = os.path.join(save_dir, file)

        df = pd.read_csv(path, index_col=0)
        df.index.name = "taxon"
        df = df.reset_index()

        if df.empty:
            continue

        if "growth_rate" not in df.columns:
            growth_candidates = [c for c in df.columns if "growth" in c.lower()]
            if not growth_candidates:
                raise ValueError(f"No growth-rate column found in {path}")
            df.rename(columns={growth_candidates[0]: "growth_rate"}, inplace=True)

        df["method"] = method
        df["tradeoff"] = tradeoff
        taxon_growth_dfs.append(df)

    if not taxon_growth_dfs:
        raise FileNotFoundError(
            f"No individual taxon growth files found matching criteria for '{com_name}'."
        )

    taxon_growth_df = pd.concat(taxon_growth_dfs, ignore_index=True)
    taxon_growth_df = taxon_growth_df[["taxon", "growth_rate", "tradeoff", "method"]].copy()
    taxon_growth_df = taxon_growth_df.dropna(subset=["taxon", "growth_rate"]).copy()

    taxon_growth_df["taxon"] = taxon_growth_df["taxon"].map(lambda x: str(x).strip())
    taxon_growth_df = taxon_growth_df[
        ~taxon_growth_df["taxon"].str.lower().isin(["medium", "community", "nan"])
    ].copy()

    if normalize_by_taxa:
        num_taxa = taxon_growth_df["taxon"].nunique()
        taxon_growth_df["growth_rate"] = taxon_growth_df["growth_rate"] * (1 / num_taxa)

    unique_methods = taxon_growth_df["method"].unique()
    palette = dict(zip(unique_methods, sns.color_palette("tab10", len(unique_methods))))

    taxa_method_order = ["pFBA", "L2pFBA"]
    method_label_map = {
        "l2pfba": "L2pFBA",
        "pfba": "pFBA",
    }

    taxon_growth_df = taxon_growth_df.copy()
    taxon_growth_df["method_label"] = taxon_growth_df["method"].str.lower().map(
        lambda m: method_label_map.get(m, m)
    )

    unique_methods = [m for m in taxa_method_order if m in taxon_growth_df["method_label"].unique()]
    if not unique_methods:
        unique_methods = list(taxon_growth_df["method_label"].unique())
    palette = dict(zip(unique_methods, sns.color_palette("tab10", len(unique_methods))))

    for tr, grp in taxon_growth_df.groupby("tradeoff"):
        grp = grp.copy()
        grp["taxon"] = grp["taxon"].astype(str)

        def taxon_sort_key(v):
            s = str(v)
            if s.isdigit():
                return (0, int(s))
            return (1, s)

        taxa_order = sorted(grp["taxon"].unique(), key=taxon_sort_key)

        plt.figure(figsize=(12, 5))
        ax = sns.barplot(
            data=grp,
            x="taxon",
            y="growth_rate",
            hue="method_label",
            order=taxa_order,
            hue_order=[m for m in taxa_method_order if m in grp["method_label"].unique()],
            palette=palette,
            dodge=True,
            errorbar=None,
        )
        ax.set_xlabel("Taxon")
        ax.set_ylabel(
            "Normalised growth rate $\mu_i$ [1/h]"
            if normalize_by_taxa
            else "Growth Rate"
        )
        ax.set_title(
            f"{com_name}: Individual Taxon Growth Across Methods (Tradeoff = {tr})"
        )
        ax.legend(title="Methods")
        plt.xticks(rotation=45, ha="right")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()

    return summary_df, taxon_growth_df

## 2. Create Communities & Load Media

In [ ]:
HvSC1_df = taxonomy_df[(taxonomy_df["Syncom"] == "HvSC1") | (taxonomy_df["Syncom"] == "Both")].copy()
HvSC1_df = HvSC1_df[["Strain ID","Family"]].rename(columns={'Strain ID': 'id', 'Family': 'family'})
HvSC1_df["id"] = HvSC1_df["id"].astype(str)
HvSC1_df["abundance"] = 1
HvSC1_df["file"] = HvSC1_df['id'].apply(lambda x: f"{model_dir1}/{x}_or_mb1_mdr_rdr_dp_mb2_lib_bz_fix.xml")

HvSC1 = Community(HvSC1_df, id = "HvSC1", name = "HvSC1")

In [ ]:
HvSC2_df = taxonomy_df[(taxonomy_df["Syncom"] == "HvSC2") | (taxonomy_df["Syncom"] == "Both")].copy()
HvSC2_df = HvSC2_df[["Strain ID","Family"]].rename(columns={'Strain ID': 'id', 'Family': 'family'})
HvSC2_df["id"] = HvSC2_df["id"].astype(str)
HvSC2_df["abundance"] = 1
HvSC2_df["file"] = HvSC2_df['id'].apply(lambda x: f"{model_dir2}/{x}_or_mb1_mdr_rdr_dp_mb2_lib_bz_fix.xml")

HvSC2 = Community(HvSC2_df, id = "HvSC2", name = "HvSC2")

In [ ]:
# Dropin community: dropin 892 to SC2
HvSC2_892_df = taxonomy_df[(taxonomy_df["Syncom"] == "HvSC2") | (taxonomy_df["Syncom"] == "Both") | (taxonomy_df["Strain ID"] == 892)].copy()
HvSC2_892_df = HvSC2_892_df[["Strain ID","Family"]].rename(columns={'Strain ID': 'id', 'Family': 'family'})
HvSC2_892_df["id"] = HvSC2_892_df["id"].astype(str)
HvSC2_892_df["abundance"] = 1
HvSC2_892_df["file"] = HvSC2_892_df['id'].apply(lambda x: f"{model_dir}/{x}_or_mb1_mdr_rdr_dp_mb2_lib_bz_fix.xml")

HvSC2_892 = Community(HvSC2_892_df, id = "HvSC2_892", name = "HvSC2_892")

In [ ]:
af_path = os.path.join(media_dir, "af_diff_growth/combined_af7_c1i127.csv")
af_complete = pd.read_csv(af_path, names = ["reaction", "flux"])
af_complete = create_medium(af_complete)

HvSC1.medium = af_complete
HvSC2.medium = af_complete
HvSC2_892.medium = af_complete

In [ ]:
opt_sol1 = HvSC1.optimize()
opt_sol2 = HvSC2.optimize()

## 3. Optimization Strategy Evaluations (Tradeoff & pFBA)

In [ ]:
fractions = [x / 10.0 for x in range(0, 11, 1)]
complete_com = cooperative_tradeoff_custom(HvSC1, opt_sol1.objective_value, fractions)
complete_com2 = cooperative_tradeoff_custom(HvSC2, opt_sol2.objective_value, fractions)
complete_com_micom = cooperative_tradeoff(HvSC1, min_growth=0.0, fluxes=False, pfba=False, fraction=fractions, atol=1e-4, rtol=1e-4)

In [ ]:
# Run pFBA evaluation
pfba_fractions = [0.5, 1.0]
results_sc1 = run_fba(HvSC1, fractions=pfba_fractions, fluxes=True)
results_sc2 = run_fba(HvSC2, fractions=pfba_fractions, fluxes=True)

save_tradeoff_results(results_sc1, save_dir, prefix="HvSC1_pfba_x")
save_tradeoff_results(results_sc2, save_dir, prefix="HvSC2_pfba_x")

## 4. Compare Results to L2 Norm Regularization

In [ ]:
opt_Sc1 = HvSC1.optimize()
opt_Sc2 = HvSC2.optimize()

cooperative_tradeoff_c1 = cooperative_tradeoff_custom(HvSC1, opt_Sc1.growth_rate, fraction=fractions, fluxes=True, atol=1e-4, rtol=1e-4)
cooperative_tradeoff_c2 = cooperative_tradeoff_custom(HvSC2, opt_Sc2.growth_rate, fraction=fractions, fluxes=True, atol=1e-4, rtol=1e-4)

save_tradeoff_results(cooperative_tradeoff_c1, save_dir, "HvSC1_L2")
save_tradeoff_results(cooperative_tradeoff_c2, save_dir, "HvSC2_L2")

In [ ]:
# Inspect exchange fluxes
c = cooperative_tradeoff_c1.iloc[0]["solution"].fluxes
c = c.loc[:, c.columns.str.startswith("EX_")]

p = pd.read_csv(os.path.join(save_dir, "HvSC1_l2pfba_10_fluxes.csv"))
p = p.loc[:, p.columns.str.startswith("EX_")]

## 5. Sugar Block Test Cases

In [ ]:
with HvSC1 as community:
    community.medium = af_complete
    e2m_rxn = [rxn for rxn in community.reactions if "EX_" in rxn.id and ("glc__D_e" in rxn.id or "cellb_e" in rxn.id)]
    fractions = [0.5]
    sol_opt_pre = community.optimize()
    print(f"optimize solution pre bloc: {sol_opt_pre.objective_value}")
    coop_sol_micom_pre = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False,  min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    coop_sol_custom_pre = cooperative_tradeoff_custom(community, fraction = fractions, min_growth =sol_opt_pre.objective_value)
    gr_pre = coop_sol_custom_pre.iloc[0]["solution"]
    print(f"Growth pre sugar block, custom: {gr_pre.growth_rate}")
    print(f"Growth pre sugar block, MICOM: {coop_sol_micom_pre.growth_rate}")


    for rxn in e2m_rxn:
        #print(f"PreBlock {rxn.id} bounds set to: {rxn.bounds}")
        community.reactions.get_by_id(rxn.id).bounds = (0, 0.0)
        #print(f"PostBlock {rxn.id} bounds set to: {rxn.bounds}")
    sol_opt = HvSC1.optimize()
    coop_sol_custom = cooperative_tradeoff_custom(community, fraction = fractions, min_growth = sol_opt.objective_value)
    coop_sol_micom = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False, min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    gr_post = coop_sol_custom.iloc[0]["solution"]
    print(f"Growth post sugar block, custom: {gr_post.growth_rate}")
    print(f"Growth post sugar block, MICOM: {coop_sol_micom.growth_rate}")

optimize solution pre bloc: 7.535269994625795
Growth pre sugar block, custom: 3.767635000673113
Growth pre sugar block, MICOM: 3.767635171176763
Growth post sugar block, custom: 3.5734187917554605
Growth post sugar block, MICOM: 3.5734194569336357


In [ ]:
with HvSC1 as community:
    community.medium = af_complete
    e2m_rxn = [rxn for rxn in community.reactions if "EX_" in rxn.id and ("glc__D_e" in rxn.id or "cellb_e" in rxn.id)]
    fractions = [0.5]
    sol_opt_pre = community.optimize()
    print(f"optimize solution pre bloc: {sol_opt_pre.objective_value}")
    coop_sol_micom_pre = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False,  min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    coop_sol_custom_pre = cooperative_tradeoff_custom(community, fraction = fractions, min_growth =sol_opt_pre.objective_value)
    gr_pre = coop_sol_custom_pre.iloc[0]["solution"]
    print(f"Growth pre sugar block, custom: {gr_pre.growth_rate}")
    print(f"Growth pre sugar block, MICOM: {coop_sol_micom_pre.growth_rate}")


    for rxn in e2m_rxn:
        #print(f"PreBlock {rxn.id} bounds set to: {rxn.bounds}")
        community.reactions.get_by_id(rxn.id).bounds = (0, 10.0)
        #print(f"PostBlock {rxn.id} bounds set to: {rxn.bounds}")
    sol_opt = HvSC1.optimize()
    coop_sol_custom = cooperative_tradeoff_custom(community, fraction = fractions, min_growth = sol_opt.objective_value)
    coop_sol_micom = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False, min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    gr_post = coop_sol_custom.iloc[0]["solution"]
    print(f"Growth post sugar block, custom: {gr_post.growth_rate}")
    print(f"Growth post sugar block, MICOM: {coop_sol_micom.growth_rate}")

optimize solution pre bloc: 7.535269994625795
Growth pre sugar block, custom: 3.767635000673113
Growth pre sugar block, MICOM: 3.767635171176763
Growth post sugar block, custom: 3.5734187718568395
Growth post sugar block, MICOM: 3.573418769979675


In [ ]:
with HvSC1 as community:
    community.medium = af_complete
    e2m_rxn = [rxn for rxn in community.reactions if "EX_" in rxn.id and ("glc__D_e" in rxn.id or "cellb_e" in rxn.id)]
    fractions = [0.5]
    sol_opt_pre = community.optimize()
    print(f"optimize solution pre bloc: {sol_opt_pre.objective_value}")
    coop_sol_micom_pre = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False,  min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    coop_sol_custom_pre = cooperative_tradeoff_custom(community, fraction = fractions, min_growth =sol_opt_pre.objective_value)
    gr_pre = coop_sol_custom_pre.iloc[0]["solution"]
    print(f"Growth pre sugar block, custom: {gr_pre.growth_rate}")
    print(f"Growth pre sugar block, MICOM: {coop_sol_micom_pre.growth_rate}")


    for rxn in e2m_rxn:
        #print(f"PreBlock {rxn.id} bounds set to: {rxn.bounds}")
        community.reactions.get_by_id(rxn.id).bounds = (-10.0, 0.0)
        #print(f"PostBlock {rxn.id} bounds set to: {rxn.bounds}")
    sol_opt = HvSC1.optimize()
    coop_sol_custom = cooperative_tradeoff_custom(community, fraction = fractions, min_growth = sol_opt.objective_value)
    coop_sol_micom = cooperative_tradeoff_nocross(community, fraction=fractions, fluxes=False, min_growth=0.0, pfba=False,atol=1e-4, rtol=1e-4 )
    gr_post = coop_sol_custom.iloc[0]["solution"]
    print(f"Growth post sugar block, custom: {gr_post.growth_rate}")
    print(f"Growth post sugar block, MICOM: {coop_sol_micom.growth_rate}")

optimize solution pre bloc: 7.535269994625795
Growth pre sugar block, custom: 3.767635000673113
Growth pre sugar block, MICOM: 3.767635171176763
Growth post sugar block, custom: 3.726160168487522
Growth post sugar block, MICOM: 3.7261647140364933
